In [ ]:
# Initialization and environment setup
%run ../../init.py
import math
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import torch
import torch.nn as nn
import torch.nn.functional as F
from ipywidgets import interact, fixed
import ipywidgets as widgets
import helpers as hp

sns.set_theme(style="whitegrid", palette="deep")

# Transformers & Self-Attention
## Eliminating Recurrence, Scaled Dot-Product, and Foundation Models

This lecture explores the Transformer architecture:
- **The Paradigm Shift**: "Attention Is All You Need" (Vaswani et al. 2017)
- **Scaled Dot-Product Attention**: Queries, Keys, Values, and the $\frac{1}{\sqrt{d_k}}$ scaling factor
- **Multi-Head Attention (MHA)**: Parallel representation subspaces
- **Positional Encodings**: Injecting sequence order without recurrence
- **Complete Transformer Block**: Multi-Head Attention, Add & Norm, and Feed-Forward Networks
- **Modern Taxonomy**: Encoder-only (BERT), Decoder-only (GPT), and Vision Transformers (ViT)

# The Paradigm Shift: Attention Is All You Need
## Eliminating Recurrence Entirely

While Bahdanau attention resolved the Seq2Seq vector bottleneck, it still relied on a sequential RNN backbone:
- Computing $\mathbf{h}_t$ requires waiting for $\mathbf{h}_{t-1}$, preventing full parallelization across time steps on modern GPU clusters.

In 2017, Ashish Vaswani et al. proposed **The Transformer**:
- Eliminates recurrence and convolutions entirely.
- Computes pairwise interactions directly between **all tokens simultaneously** via **Self-Attention**.
- Path length between any two tokens in a sequence is strictly **$\mathcal{O}(1)$**, enabling massively parallel training on billions of tokens.

<small>📖 *Source: [Bishop, C. M., & Bishop, H. (2024). Deep Learning: Foundations and Concepts. Springer (Section 12.1)](https://doi.org/10.1007/978-3-031-45468-4)*</small>

# Scaled Dot-Product Attention
## Queries, Keys, and Values

For an input sequence embedded as matrix $\mathbf{X} \in \mathbb{R}^{T \times D}$, we project into three distinct matrices:
- **Queries ($\mathbf{Q}$)**: What each token is looking for ($\mathbf{Q} = \mathbf{X} \mathbf{W}^Q$)
- **Keys ($\mathbf{K}$)**: What each token contains / advertises ($\mathbf{K} = \mathbf{X} \mathbf{W}^K$)
- **Values ($\mathbf{V}$)**: The actual informational content ($\mathbf{V} = \mathbf{X} \mathbf{W}^V$)

The **Scaled Dot-Product Attention** computes:
$$\mathrm{Attention}(\mathbf{Q}, \mathbf{K}, \mathbf{V}) = \mathrm{softmax}\left(\frac{\mathbf{Q}\mathbf{K}^\mathrm{T}}{\sqrt{d_k}}\right) \mathbf{V}$$

where:
- $\mathbf{A} = \frac{\mathbf{Q}\mathbf{K}^\mathrm{T}}{\sqrt{d_k}} \in \mathbb{R}^{T \times T}$ computes pairwise compatibility scores
- $\mathrm{softmax}(\mathbf{A})$ converts raw scores into probability distributions over keys (each row sums to 1)
- The output is a weighted sum of value vectors $\mathbf{V}$.

## The Scaling Factor $\frac{1}{\sqrt{d_k}}$
## Preventing Gradient Saturation

Why is the dot product divided by $\sqrt{d_k}$?

Assume query components $q_i$ and key components $k_i$ are independent random variables with zero mean and unit variance:
$$\mathbb{E}[q_i] = 0, \quad \mathrm{var}[q_i] = 1, \quad \mathbb{E}[k_i] = 0, \quad \mathrm{var}[k_i] = 1$$

Their dot product $q^\mathrm{T} k = \sum_{i=1}^{d_k} q_i k_i$ has:
$$\mathbb{E}\left[\mathbf{q}^\mathrm{T}\mathbf{k}\right] = 0, \quad \mathrm{var}\left[\mathbf{q}^\mathrm{T}\mathbf{k}\right] = \sum_{i=1}^{d_k} \mathrm{var}[q_i k_i] = d_k$$

- For large projection dimensions (e.g. $d_k = 64$ or $128$), the dot products grow very large in magnitude (variance $d_k$).
- Feeding large values into the softmax function pushes it into extreme saturation regions where gradients vanish ($S'(z) \approx 0$).
- Scaling by $\frac{1}{\sqrt{d_k}}$ renormalizes the variance strictly back to **$1$**, ensuring robust gradient propagation!

# Multi-Head Attention (MHA)
## Attending to Distinct Representation Subspaces

Instead of performing a single attention function with dimension $d$, **Multi-Head Attention** linearly projects queries, keys, and values $h$ times with distinct learned parameter matrices:

$$\mathrm{head}_i = \mathrm{Attention}\left(\mathbf{Q}\mathbf{W}_i^Q, \mathbf{K}\mathbf{W}_i^K, \mathbf{V}\mathbf{W}_i^V\right), \quad i = 1, \dots, h$$

The independent heads are concatenated and projected:
$$\mathrm{MultiHead}(\mathbf{Q}, \mathbf{K}, \mathbf{V}) = \left[\mathrm{head}_1, \dots, \mathrm{head}_h\right] \mathbf{W}^O$$

## Architecture of Multi-Head Projections

```
   Q, K, V
      |
   +--+--+--+ (h parallel heads)
   |  |  |  |
  [Head 1] [Head 2] ... [Head h]  (e.g. Head 1 tracks syntax, Head 2 tracks coreference)
   |  |  |  |
   +--+--+--+
      |
  [ Concat ] ---> [ Linear W^O ] ---> Output
```

Multi-head attention allows the model to jointly attend to information from different representation subspaces at different positions.

## Positional Encodings
## Breaking Permutation Equivariance

Unlike RNNs, the Self-Attention operation is strictly **permutation equivariant**:
$$\mathrm{Attention}(\mathbf{P}\mathbf{X}) = \mathbf{P} \mathrm{Attention}(\mathbf{X})$$
If we shuffle the words in a sentence, the attention outputs shuffle identically, with zero awareness of word order!

To inject sequence order, Vaswani et al. add fixed **Sinusoidal Positional Encodings** $\mathbf{P} \in \mathbb{R}^{T \times D}$ directly to the input embeddings:
$$\tilde{\mathbf{X}} = \mathbf{X} + \mathbf{P}$$

$$\mathbf{P}_{pos, 2i} = \sin\left(\frac{pos}{10000^{2i/D}}\right), \quad \mathbf{P}_{pos, 2i+1} = \cos\left(\frac{pos}{10000^{2i/D}}\right)$$

This allows the model to attend to relative positions because for any fixed offset $k$, $\mathbf{P}_{pos+k}$ can be represented as a linear function of $\mathbf{P}_{pos}$.

# The Full Transformer Architecture
## Blocks, Normalization, and Residuals

A standard Transformer layer consists of two sub-layers wrapped with residual connections and Layer Normalization:

1. **Sub-Layer 1: Multi-Head Self-Attention**:
   $$\mathbf{Z}^{(l)} = \mathrm{LayerNorm}\left(\mathbf{X}^{(l-1)} + \mathrm{MultiHead}(\mathbf{X}^{(l-1)})\right)$$

2. **Sub-Layer 2: Position-Wise Feed-Forward Network (FFN)**:
   Applied to each position independently:
   $$\mathrm{FFN}(\mathbf{Z}) = \max\left(0, \mathbf{Z}\mathbf{W}_1 + \mathbf{b}_1\right) \mathbf{W}_2 + \mathbf{b}_2$$
   $$\mathbf{X}^{(l)} = \mathrm{LayerNorm}\left(\mathbf{Z}^{(l)} + \mathrm{FFN}(\mathbf{Z}^{(l)})\right)$$

In auto-regressive generation (e.g. GPT), **Causal Masking** sets upper-triangular attention logits to $-\infty$ to prevent attending to future tokens.

In [ ]:
# Clean PyTorch Implementation of Scaled Dot-Product Attention
def scaled_dot_product_attention(Q, K, V, mask=None):
    d_k = Q.size(-1)
    scores = torch.matmul(Q, K.transpose(-2, -1)) / math.sqrt(d_k)
    
    if mask is not None:
        scores = scores.masked_fill(mask == 0, -1e9)
        
    weights = F.softmax(scores, dim=-1)
    output = torch.matmul(weights, V)
    return output, weights

# Verification with sample sequence (Batch=1, Seq_len=4, D_k=8)
torch.manual_seed(42)
Q = torch.randn(1, 4, 8)
K = torch.randn(1, 4, 8)
V = torch.randn(1, 4, 8)

out, attn_weights = scaled_dot_product_attention(Q, K, V)
print("Output Shape       :", out.shape)
print("Attention Weights  :\n", attn_weights.squeeze().detach().numpy().round(3))
print("Row Sum Verification:", attn_weights.squeeze().sum(dim=-1).detach().numpy().round(3))

In [ ]:
# Interactive Attention Heatmap Visualization Helper
def plot_attention_matrix(apply_causal_mask=False):
    tokens = ["Deep", "Learning", "Attention", "Transformer"]
    T = len(tokens)
    
    torch.manual_seed(10)
    q = torch.randn(1, T, 8)
    k = torch.randn(1, T, 8)
    v = torch.randn(1, T, 8)
    
    mask = torch.tril(torch.ones(T, T)) if apply_causal_mask else None
    _, weights = scaled_dot_product_attention(q, k, v, mask=mask)
    
    plt.figure(figsize=(5.5, 4.2))
    sns.heatmap(weights.squeeze().detach().numpy(), annot=True, fmt=".2f", cmap="Blues",
                xticklabels=tokens, yticklabels=tokens, cbar=False)
    plt.title(f"Self-Attention Weights (Causal Mask: {apply_causal_mask})", fontsize=11)
    plt.xlabel("Key Tokens (Content)")
    plt.ylabel("Query Tokens (Focus)")
    plt.tight_layout()
    plt.show()

In [ ]:
# Interactive exploration: Bidirectional vs Causal (Auto-regressive) Attention
interact(plot_attention_matrix, apply_causal_mask=[False, True]);

# Summary: Transformers & Foundation Models

1. **Self-Attention Mechanism**:
   - Replaces sequential recurrence with pairwise token routing $\mathrm{softmax}(\mathbf{Q}\mathbf{K}^\mathrm{T} / \sqrt{d_k})\mathbf{V}$.
   - Eliminates the vanishing gradient bottleneck with direct $\mathcal{O}(1)$ information flow.

2. **Multi-Head Parallelism**:
   - Computes attention across $h$ parallel subspaces, allowing concurrent capture of syntactic and semantic relations.

3. **Modern Foundation Model Taxonomy**:
   - **Encoder-only (BERT)**: Bidirectional context for classification and embeddings.
   - **Decoder-only (GPT)**: Causal masked autoregression for generative language modeling.
   - **Vision Transformers (ViT)**: Applying self-attention directly to flattened image patches.